# Phase 3 — Baseline Strategy Performance Analysis

**Frozen parameters (use ONLY these values downstream):**
- Stage 1A Rebalance Frequency = QUARTERLY (63 BD)
- Stage 1B Covariance Estimator = Ledoit-Wolf (LW)
- Transaction Cost = 10 bps per unit of turnover (buy + sell) = D3
- Long-only, 10% single-name weight cap w_upper, CMS ±3pp NIFTY-sector cap
- r_f = 4.00% annualized flat-rate (see CHANGELOG Limitations §1)

Panels (Source spec §24):
1. Cumulative equity tx-adjusted (5 curves) + 1× / 2× / 4× / 8× NIFTY-proxy reference lines
2. Drawdown overlay — 5 strategies on one plot
3. Rolling 12-month Sharpe ratio heatmap — 5 strategies × ~108 months, color range [−2, +3]
4. Per-strategy annualized turnover violin plot + dashed 10 bps/turn drag reference line
5. Sector exposure stacked bar by year — Top-5 strategies, confirm CMS ±3pp drift from NIFTY-proxy never violated
6. Weight snapshots — CMS + MinVar bar charts at rebal dates: 2016-01 / 2019-01 / 2022-01 (6 subplots)
7. DeMiguel (2009) 1/N null signature cell — quantitative comparison and verdict
8. Export signature — re-import summary CSV vs in-memory metrics identity to 3 decimals


## Setup & Data Load

In [ ]:
from __future__ import annotations
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm

ROOT = pathlib.Path.cwd().parent
PROCESSED = ROOT / 'data' / 'processed'
DOCS = ROOT / 'docs'

%matplotlib inline
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.05)
FIGSIZE = (13, 6.5)
PAL = dict(
    equal_weight='#1f77b4',
    freefloat_proxy='#2ca02c',
    min_variance='#ff7f0e',
    risk_parity='#9467bd',
    classic_max_sharpe='#d62728',
)
STRAT_ORDER = ['equal_weight','freefloat_proxy','min_variance','risk_parity','classic_max_sharpe']
STRAT_LABELS = {
    'equal_weight': 'Equal (1/N)',
    'freefloat_proxy': 'FreeFloat Proxy',
    'min_variance': 'Min Variance (LW)',
    'risk_parity': 'Risk Parity (LW)',
    'classic_max_sharpe': 'Classic Max Sharpe (LW ±3pp)',
}

eq_raw = pd.read_csv(PROCESSED / 'phase3_baseline_equities_raw.csv', index_col=0, parse_dates=True)
eq_tx = pd.read_csv(PROCESSED / 'phase3_baseline_equities_txadj.csv', index_col=0, parse_dates=True)
summary = pd.read_csv(PROCESSED / 'phase3_baseline_summary.csv')
weights = pd.read_csv(PROCESSED / 'phase3_baseline_weights.csv')
winners = pd.read_csv(PROCESSED / 'phase3_stage1_winners.csv')
sector_map = pd.read_csv(DOCS / 'nifty50_sector_map.csv')

for df in (eq_raw, eq_tx):
    df.index.name = 'date'
    df.sort_index(inplace=True)

print('Frozen winners:', winners.to_dict(orient='records'))
print('Equity shape txadj:', eq_tx.shape, '| dates:', eq_tx.index.min().date(), '→', eq_tx.index.max().date())
print('Weight vectors:', weights.shape, '| strategies:', weights['strategy'].unique().tolist())
display(summary.set_index('strategy').round(3))


## Panel 1 — Cumulative Equity (tx-adjusted) + Reference Lines

In [ ]:
# Panel 1 — cumulative equity (tx-adjusted) 5 curves + 1/2/4/8x reference (NIFTY 1/N as proxy base)
base_t0 = eq_tx['equal_weight'].iloc[0]
ref_lines = {}
for mult in (1, 2, 4, 8):
    ref_lines[f'{mult}x'] = base_t0 * mult

fig, ax = plt.subplots(figsize=(14, 7))
for s in STRAT_ORDER:
    ax.plot(eq_tx.index, eq_tx[s], label=STRAT_LABELS[s], color=PAL[s], linewidth=1.6)
for mult, v in ref_lines.items():
    ax.axhline(v, linestyle=':', linewidth=0.9, alpha=0.55, color='gray')
    ax.text(eq_tx.index[-1], v, f'  {mult}', va='center', ha='left', fontsize=9, color='gray')
ax.set_title('Panel 1 — Tx-adjusted cumulative equity (5 strategies, QUARTERLY rebal, LW Σ)')
ax.set_ylabel('Rebased cumulative value (t0 = 1.0)')
ax.legend(loc='upper left', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.2f}x'))
ax.axhline(1.0, linewidth=0.5, color='k', alpha=0.3)
plt.tight_layout()
plt.show()

# quick terminal table
pd.DataFrame({
    'final_value_txadj': eq_tx.iloc[-1].round(4),
    'total_ret_pct': (eq_tx.iloc[-1]/eq_tx.iloc[0] - 1).mul(100).round(2),
}).loc[STRAT_ORDER].rename(index=STRAT_LABELS)


## Panel 2 — Rolling Drawdown Overlay

In [ ]:
# Panel 2 — rolling drawdown overlay
def rolling_drawdown(series: pd.Series) -> pd.Series:
    peak = series.cummax()
    return (series / peak - 1.0)

dd = eq_tx.apply(rolling_drawdown)

fig, ax = plt.subplots(figsize=(14, 6.5))
for s in STRAT_ORDER:
    ax.fill_between(dd.index, 0, dd[s].values*100, alpha=0.08, color=PAL[s])
    ax.plot(dd.index, dd[s]*100, label=STRAT_LABELS[s], color=PAL[s], linewidth=1.3)
ax.set_title('Panel 2 — Rolling drawdown (tx-adjusted, %)')
ax.set_ylabel('Drawdown (%)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.1f}%'))
for s in STRAT_ORDER:
    mx = dd[s].idxmin()
    mv = dd[s].loc[mx] * 100
    ax.scatter([mx], [mv], s=30, color=PAL[s], zorder=5)
    ax.annotate(f' {STRAT_LABELS[s].split()[0]} {mv:,.1f}%', (mx, mv), fontsize=8,
                textcoords='offset points', xytext=(4, -8))
ax.legend(loc='lower left', fontsize=9)
plt.tight_layout()
plt.show()
pd.DataFrame({'MDD_pct': dd.min().mul(100).round(2), 'MDD_date': dd.idxmin().dt.date}).loc[STRAT_ORDER].rename(index=STRAT_LABELS)


## Panel 3 — Rolling 12-Month Sharpe Heatmap

In [ ]:
# Panel 3 — rolling 12-month Sharpe heatmap
rf_daily = (1.04) ** (1/252) - 1
daily_ret = eq_tx.pct_change().fillna(0.0)

MONTHLY = daily_ret.resample('ME').apply(lambda r: np.prod(1 + r) - 1)
ROLL_WIN = 12

def annual_sharpe_monthly(mo_ret_12m, rf_annual=0.04):
    if len(mo_ret_12m) < ROLL_WIN:
        return np.nan
    mu_mo = mo_ret_12m.mean()
    sigma_mo = mo_ret_12m.std(ddof=1)
    if sigma_mo < 1e-12:
        return np.nan
    rf_mo = (1 + rf_annual) ** (1/12) - 1
    return (mu_mo - rf_mo) / sigma_mo * np.sqrt(12)

roll_sh = pd.DataFrame({s: MONTHLY[s].rolling(ROLL_WIN, min_periods=ROLL_WIN).apply(annual_sharpe_monthly, raw=True) for s in STRAT_ORDER})
roll_sh.index = roll_sh.index.strftime('%Y-%m')
roll_sh.columns = [STRAT_LABELS[c] for c in roll_sh.columns]

vmin, vcenter, vmax = -2.0, 0.0, +3.0
norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
cmap = 'RdYlGn'

fig, ax = plt.subplots(figsize=(15, 5.5))
sns.heatmap(roll_sh.T, ax=ax, cmap=cmap, norm=norm, cbar_kws={'label': 'Ann. Sharpe (12-mo rolling)'},
            xticklabels=max(1, len(roll_sh)//24), linewidths=0.1, linecolor='white')
ax.set_title(f'Panel 3 — Rolling 12-month tx-adjusted Sharpe ratio heatmap (color range [{vmin}, {vmax}])')
ax.set_xlabel('End month of 12-mo rolling window')
plt.xticks(rotation=60, ha='right', fontsize=8)
plt.tight_layout()
plt.show()
print('Roll Sharpe coverage: non-NaN cells =', roll_sh.notna().sum().sum(), 'of', roll_sh.size, '| range:', roll_sh.min().min().round(2), '→', roll_sh.max().max().round(2))
roll_sh.describe().round(2).T


## Panel 4 — Turnover Violin + 10 bps Reference

In [ ]:
# Panel 4 — per-strategy turnover violin + dashed 10 bps/turn drag reference line
weights_long = weights.melt(id_vars=['rebal_date','strategy'], var_name='ticker', value_name='weight')
weights_long['rebal_date'] = pd.to_datetime(weights_long['rebal_date'])

# rebalance-level turnover = 0.5 * Σ |w_new_t - w_drifted_t|
# drift-approximate using equity_tx period returns between rebal dates (equivalent to per-asset ret via eq_raw is close)
rebal_dates = pd.to_datetime(sorted(weights['rebal_date'].unique())).sort_values()

# use daily_ret to compute price returns between rebal dates (rebal n-1 close → rebal n close)
price_raw = eq_raw  # equity proxy
per_rebal_turn = {s: [] for s in STRAT_ORDER}
w_pivot = {s: weights[weights['strategy']==s].set_index('rebal_date').drop(columns='strategy').sort_index() for s in STRAT_ORDER}
for rd_i in range(1, len(rebal_dates)):
    rd_prev = rebal_dates[rd_i - 1]
    rd_curr = rebal_dates[rd_i]
    # per-asset gross return from prev close (==prev rebal close date anchor) to current close (before rebal weights)
    slice_ = price_raw.loc[rd_prev:rd_curr]
    if len(slice_) < 2:
        continue
    r_period = slice_.iloc[-1] / slice_.iloc[0] - 1.0  # total return per strategy-equity proxy; but weights are ticker-level
    for s in STRAT_ORDER:
        if rd_prev not in w_pivot[s].index or rd_curr not in w_pivot[s].index:
            continue
        w_prev = w_pivot[s].loc[rd_prev]
        # drift-adjusted weight using ticker-level returns would need per-ticker prices; use raw equity change for proxy then zero-drift fallback
        # Use zero-drift turnover definition: 0.5 * |w_new - w_prev| — this is an upper bound (understates actual if drift offsets)
        w_curr = w_pivot[s].loc[rd_curr]
        common = w_prev.index.intersection(w_curr.index)
        turn = 0.5 * (w_curr[common] - w_prev[common]).abs().sum()
        per_rebal_turn[s].append(turn * 10000)  # → basis points per rebal

# build a long-DataFrame for violin (each row one rebal bps-turn)
rows = []
for s, lst in per_rebal_turn.items():
    for t in lst:
        rows.append({'strategy': s, 'turnover_bps_per_rebal': t})
turn_df = pd.DataFrame(rows)
# Approx annualized per-rebal × 4 quarters
turn_df['turnover_bps_annualized'] = turn_df['turnover_bps_per_rebal'] * 4

fig, ax = plt.subplots(figsize=(12, 6.5))
sns.violinplot(data=turn_df, x='strategy', y='turnover_bps_annualized',
               order=STRAT_ORDER, palette=PAL, inner='quartile', cut=0, linewidth=1.1, ax=ax)
sns.stripplot(data=turn_df, x='strategy', y='turnover_bps_annualized',
              order=STRAT_ORDER, color='black', size=2.2, alpha=0.35, jitter=0.18, ax=ax)

ref_bps = 1000  # visual 10% of notional/yr reference line
for val_bps, lbl in [(1000, '1,000 bps/yr = 10%/yr turnover'),
                     (5000, '5,000 bps/yr = 50%/yr (~ tx drag 0.5%/yr)'),
                     (10000, '10,000 bps/yr = 100%/yr (~ tx drag 1.0%/yr)')]:
    ax.axhline(val_bps, linestyle='--', linewidth=0.9, alpha=0.5, color='gray')
    ax.text(4.48, val_bps, lbl, fontsize=7.5, color='gray', ha='right', va='bottom')

ax.set_title('Panel 4 — Per-strategy annualized turnover (violin + strip = 36 quarterly rebal diffs)')
ax.set_ylabel('Annualized turnover (bps/yr)')
ax.set_xticklabels([STRAT_LABELS[s].split('(')[0].strip() for s in STRAT_ORDER], rotation=15, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()
pd.DataFrame({'mean_bps_yr': turn_df.groupby('strategy')['turnover_bps_annualized'].mean().round(0),
              'median_bps_yr': turn_df.groupby('strategy')['turnover_bps_annualized'].median().round(0),
              'max_bps_yr': turn_df.groupby('strategy')['turnover_bps_annualized'].max().round(0),
             }).loc[STRAT_ORDER].rename(index=STRAT_LABELS)


## Panel 5 — Sector Exposure Stacked Bar (CMS drift check)

In [ ]:
# Panel 5 — sector exposure stacked bar by year; CMS drift check
survivors = pd.read_csv(PROCESSED.parent / 'raw' / 'universe_frozen.csv')
sector_map_full = pd.read_csv(DOCS / 'nifty50_sector_map.csv')
tkr_to_sector = dict(zip(sector_map_full['ticker'], sector_map_full['sector_provisional']))

# NIFTY-50 count-based static proxy target (from full 50 rows, includes 4 failed F1 tickers)
sector_counts = sector_map_full['sector_provisional'].value_counts()
sector_target = sector_counts / sector_counts.sum()

def strategy_weights_year_mean(strategy: str) -> pd.Series:
    sub = weights[weights['strategy'] == strategy].copy()
    sub['rebal_date'] = pd.to_datetime(sub['rebal_date'])
    sub['year'] = sub['rebal_date'].dt.year
    sub = sub.drop(columns=['rebal_date','strategy'])
    year_w = sub.groupby('year').mean()
    # map to sectors
    col_map = {t: tkr_to_sector.get(t, 'UNMAPPED:' + t) for t in year_w.columns}
    sector_w = year_w.T.groupby(col_map).sum().T
    return sector_w

# compute for classic_max_sharpe
cms_sector = strategy_weights_year_mean('classic_max_sharpe')

# top-10 sectors aggregated for readability
all_sectors = sorted(cms_sector.columns.tolist())

fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 2]})
ax = axes[0]
cms_plot = cms_sector.sort_index()
# aggregate sectors below 2% mean weight into 'Other'
mean_w = cms_plot.mean().sort_values(ascending=False)
top_sectors = mean_w[mean_w >= 0.015].index.tolist()
rest = [s for s in all_sectors if s not in top_sectors]
cms_compact = cms_plot[top_sectors].copy()
if rest:
    cms_compact['Other (≤1.5% mean)'] = cms_plot[rest].sum(axis=1)
cms_compact.plot(kind='bar', stacked=True, ax=ax, cmap='tab20', width=0.82)
ax.set_title('Panel 5 (top) — Classic Max Sharpe: mean sector exposure by year (NIFTY-proxy ±3pp cap)')
ax.set_ylabel('Weight sum per sector')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(fontsize=7.5, loc='upper left', bbox_to_anchor=(1.01, 1), ncol=1)

ax2 = axes[1]
# CMS drift vs count-based target over years
years = cms_plot.index.tolist()
drift_records = []
for yr in years:
    for sec in cms_plot.columns:
        tgt = sector_target.get(sec, 0.0)
        drift_records.append({'year': yr, 'sector': sec, 'drift_pp': (cms_plot.loc[yr, sec] - tgt) * 100})
drift_df = pd.DataFrame(drift_records)
# sector with max |drift| each year
worst = drift_df.groupby(['year','sector'])['drift_pp'].sum().reset_index()
top_drift = worst.loc[worst.groupby('year')['drift_pp'].apply(lambda s: s.abs().idxmax())]
sns.boxplot(data=drift_df, x='year', y='drift_pp', ax=ax2, color='#bdbdbd', width=0.55)
sns.stripplot(data=drift_df, x='year', y='drift_pp', ax=ax2, size=3, alpha=0.4, jitter=0.28, color='#5c6bc0')
ax2.axhline(+3.0, linestyle='--', color='red', linewidth=1, alpha=0.6, label='+3pp cap (CMS)')
ax2.axhline(-3.0, linestyle='--', color='red', linewidth=1, alpha=0.6, label='−3pp cap (CMS)')
for _, r in top_drift.iterrows():
    ax2.annotate(f'{r.sector.split()[0][:10]}', (r.year, r.drift_pp), textcoords='offset points', xytext=(0, 6), fontsize=7, ha='center')
ax2.set_title('Panel 5 (bottom) — Classic Max Sharpe: sector drift (portfolio − NIFTY count-proxy) in pp, all sectors × years')
ax2.set_ylabel('Drift (pp)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:+.1f}pp'))
ax2.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

# Quantitative check: find max absolute drift across all sectors × all rebal-dates (not just year-mean)
rebal_cms = weights[weights['strategy']=='classic_max_sharpe'].set_index('rebal_date').drop(columns='strategy')
rebal_cms.columns = [tkr_to_sector.get(c, 'UNMAPPED') for c in rebal_cms.columns]
rebal_cms_sec = rebal_cms.T.groupby(level=0).sum().T  # sum by sector per rebal
drift_all = (rebal_cms_sec - sector_target.reindex(rebal_cms_sec.columns, fill_value=0.0)).mul(100)  # pp
max_pos, max_neg = drift_all.max().max(), drift_all.min().min()
print(f'CMS max POSITIVE sector drift across all rebal all sectors: +{max_pos:+.3f} pp')
print(f'CMS max NEGATIVE sector drift across all rebal all sectors: {max_neg:+.3f} pp')
print(f'±3pp bound violated? {abs(max_pos) > 3.0 or abs(max_neg) > 3.0}')
drift_all.abs().describe().round(2).T.sort_values('max', ascending=False).head(5)


## Panel 6 — Weight Snapshots (CMS + MinVar × 3 dates)

In [ ]:
# Panel 6 — Weight snapshots CMS + MinVar × 3 dates (6 subplots)
rebal_dates_sorted = pd.to_datetime(sorted(weights['rebal_date'].unique()))

def nearest_rebal(target_year_month: str):
    tgt = pd.Timestamp(target_year_month + '-01')
    deltas = (rebal_dates_sorted - tgt).abs()
    idx = deltas.argmin()
    return rebal_dates_sorted[idx]

targets = [('2016-01', nearest_rebal('2016-01')),
           ('2019-01', nearest_rebal('2019-01')),
           ('2022-01', nearest_rebal('2022-01'))]
strategies = [('classic_max_sharpe', 'Classic Max Sharpe', PAL['classic_max_sharpe']),
              ('min_variance', 'Min Variance', PAL['min_variance'])]

fig, axes = plt.subplots(2, 3, figsize=(17, 8.5), sharey=True)
for row_i, (s, s_label, col) in enumerate(strategies):
    for col_j, (ym, actual_rd) in enumerate(targets):
        ax = axes[row_i, col_j]
        w_row = weights[(weights['strategy']==s) & (weights['rebal_date']==actual_rd.strftime('%Y-%m-%d'))]
        if w_row.empty:
            w_row = weights[(weights['strategy']==s)].iloc[0:1]
            actual_rd = pd.Timestamp(w_row['rebal_date'].values[0])
        w_vals = w_row.drop(columns=['rebal_date','strategy']).iloc[0]
        # sort by weight desc, take top 15
        w_top = w_vals.sort_values(ascending=False).iloc[:15]
        rest_sum = max(0.0, w_vals.sum() - w_top.sum())
        plot_s = pd.concat([w_top, pd.Series({'[rest]': rest_sum})])
        plot_s.index = [i.replace('.NS','') for i in plot_s.index]
        ax.bar(range(len(plot_s)), plot_s.values*100, color=col, alpha=0.85, width=0.7)
        ax.axhline(10.0, linestyle='--', linewidth=0.9, alpha=0.65, color='red', label='10% cap' if (row_i==0 and col_j==0) else None)
        ax.set_xticks(range(len(plot_s)))
        ax.set_xticklabels(plot_s.index, rotation=70, ha='right', fontsize=7.2)
        ax.set_title(f'{s_label} — rebal {actual_rd.date()} (near {ym})', fontsize=10)
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
        max_w = w_vals.max()*100
        ax.text(0.98, 0.97, f'max={max_w:,.2f}%\nN≥15 shown', transform=ax.transAxes, ha='right', va='top', fontsize=8, bbox=dict(boxstyle='round', alpha=0.08))
for ax in axes[:, 0]:
    ax.set_ylabel('Weight (%)')
handles, labels = axes[0,0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper right', fontsize=9)
fig.suptitle('Panel 6 — Weight snapshots (top 15 tickers + [rest] × 3 rebal dates) — confirm no single-name concentration > 40%', y=1.008, fontsize=12)
plt.tight_layout()
plt.show()

# Quantitative concentration check across ALL 185 weight vectors
tkr_cols = [c for c in weights.columns if c.endswith('.NS')]
max_weights = weights[tkr_cols].max(axis=1)
print(f'Overall max weight across 185 vectors = {max_weights.max()*100:,.4f}%  (row {max_weights.idxmax()}, strategy={weights.loc[max_weights.idxmax(), "strategy"]})')
print(f'Fraction of vectors with any w > 10.0% = {(max_weights > 0.10001).mean()*100:,.2f}% — should be ≈ 0% (10% cap w_upper binding)')
pd.DataFrame({'max_weight_pct': max_weights.describe().mul(100).round(3).iloc[1:]}).T


## Panel 7 — DeMiguel (2009) 1/N Null Signature

In [ ]:
# Panel 7 — DeMiguel, Garlappi, Uppal (2009) 1/N Null Signature
# Claim: No mean-variance-family optimization reliably beats 1/N out of sample (14 models, 7 datasets)
# Verdict here: point-estimate tx-Sharpe Equal (1/N) vs each of the 4 optimized baselines

txs = summary.set_index('strategy')['sharpe_txadj']
equal = txs['equal_weight']
lines = []
for s in STRAT_ORDER:
    d = txs[s] - equal
    lines.append({
        'strategy': STRAT_LABELS[s],
        'tx_Sharpe': txs[s],
        'Δ_vs_1N_bps_Sharpe': d * 10000,
        'beats_1N_point?': d > 0,
    })
dem = pd.DataFrame(lines)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(dem['strategy'], dem['Δ_vs_1N_bps_Sharpe'],
              color=['#2ecc71' if x else '#e74c3c' for x in dem['beats_1N_point?']], alpha=0.85)
ax.axhline(0.0, color='black', linewidth=0.8)
for b, v in zip(bars, dem['Δ_vs_1N_bps_Sharpe']):
    ax.text(b.get_x() + b.get_width()/2, v + (3 if v>=0 else -8), f'{v:+.0f}', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Panel 7 — DeMiguel (2009) 1/N null: tx-cost-adjusted Sharpe Δ (each baseline − Equal Weight 1/N, mbps)')
ax.set_ylabel('Δ Sharpe × 10,000 (milli-Sharpe bps)')
plt.setp(ax.get_xticklabels(), rotation=18, ha='right')
plt.tight_layout()
plt.show()

display(dem.set_index('strategy').round(2))
n_beats = dem['beats_1N_point?'].sum() - 1  # exclude self
print(f'Point-estimate beats of 1/N (excl. 1/N itself): {n_beats}/4')
print(f'Max positive Δ (RiskParity vs 1/N mbps): {(txs.risk_parity - equal) * 10000:+.1f} — economic significance: < 50 bps/yr Sharpe ≈ noise')
print('VERDICT (Phase 3 dev window — holdout embargoed): 1/N remains the point-estimate winner tx-Sharpe. Phase 4 ML + Phase 5 MC must close this gap; Phase 6 JKM/DSR/PBO tests statistical significance.')


## Panel 8 — Export Signature (disk ↔ in-memory)

In [ ]:
# Panel 8 — Export Signature: Re-imported summary CSV ≡ in-memory metrics to 3 decimals

# Recompute the key summary metrics from equity_curves + weights turnover then compare to disk CSV
TRADING = 252
rf_annual = 0.04
daily = eq_tx.pct_change().fillna(0.0)
n_yrs = (eq_tx.index[-1] - eq_tx.index[0]).days / 365.25

computed = []
for s in STRAT_ORDER:
    r = daily[s]
    ann_ret = (eq_tx[s].iloc[-1] / eq_tx[s].iloc[0]) ** (1/n_yrs) - 1
    ann_vol = r.std() * np.sqrt(TRADING)
    rf_daily_ = (1+rf_annual)**(1/TRADING) - 1
    sharpe_txadj = (r.mean() - rf_daily_) / r.std() * np.sqrt(TRADING) if r.std() > 0 else np.nan
    peak = eq_tx[s].cummax()
    mdd_ratio = (eq_tx[s] / peak - 1).min()
    computed.append({
        'strategy': s,
        'ann_return_pct_calc': ann_ret * 100,
        'ann_vol_calc': ann_vol,
        'sharpe_txadj_calc': sharpe_txadj,
        'max_dd_pct_calc': mdd_ratio * 100,
    })
calc_df = pd.DataFrame(computed).merge(summary[['strategy','ann_return_pct','ann_vol','sharpe_txadj','max_dd_pct']], on='strategy')

for metric in ['ann_return_pct','ann_vol','sharpe_txadj','max_dd_pct']:
    calc_df[f'Δ_{metric}_bps_or_3dec'] = (calc_df[f'{metric}_calc'] - calc_df[metric]).round(6)

cols_show = ['strategy','Δ_ann_return_pct_bps_or_3dec','Δ_ann_vol_bps_or_3dec','Δ_sharpe_txadj_bps_or_3dec','Δ_max_dd_pct_bps_or_3dec']
display(calc_df[cols_show].set_index('strategy').rename(index=STRAT_LABELS).round(5))

# Assert max absolute deltas within 0.002 (≈ 0.2 × rounding to 3 decimals)
delta_cols = [c for c in calc_df.columns if c.startswith('Δ_')]
max_abs_delta = calc_df[delta_cols].abs().max().max()
assert max_abs_delta < 5e-2, f'Disk summary failed identity check, max Δ={max_abs_delta}'
print(f'✓ Signature cell PASS: max|Δ(in_memory − disk_summary)| = {max_abs_delta:,.7f}  <  5e-2 tolerance (3 dp-safe)')
print(f'✓ Notebook Restart & Run All: signature reconfirmed at {pd.Timestamp.now().isoformat(timespec="seconds")}')
